# Part 0 - Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import integrate

# constants
g     = 9.81          # m/s²
rho   = 998.2         # kg/m³
rho_s = 2650.0        # kg/m³
nu    = 1.00e-6       # m²/s   ← 20°C, NOT 10°C
nu10  = 1.31e-6       # m²/s   ← 10°C reference (for Boguchwal-Southard)
kappa = 0.41
s_sg  = (rho_s - rho) / rho   # submerged specific gravity ≈ 1.655

# Cedar Creek parameters
D50   = 0.30e-3       # m  (0.30 mm)
D90   = 0.55e-3       # m
S     = 0.0003
W     = 75.0          # m
n_man = 0.025
tau_star_c = 0.0495      # Wong-Parker critical Shields stress
k_s     = 2.5 * D50   # Einstein roughness height

### First, let's compute a depth-discharge table, so that we can figure out the bed shear stress at different discharges

In [ ]:
# pre-computed depth-discharge table (Manning n=0.4, wide channel)
# H = (Q * n / (W * S**0.5)) ** (3/5)
Q_vals = np.array([25, 50, 75, 100, 150, 200, 250, 300, 400, 500])  # m³/s
H_vals = (Q_vals * n_man / (W * S**0.5)) ** (3/5)                  # m
U_vals = Q_vals / (W * H_vals)                                      # m/s
tau_b  = rho * g * H_vals * S                                       # Pa

#### Quick sanity check to make sure our work above is correct

In [ ]:
Q_bf, H_bf, U_bf = 250.0, H_vals[6], U_vals[6]
print(f'Bankfull (Q=250):  H = {H_bf:.2f} m,  U = {U_bf:.2f} m/s,  τ_b = {U_bf:.1f} Pa')

# Part 1 -- Bedform Regime
Before computing transport, we need to know what the bed looks like. Is it flat? Rippled? Duned? The answer determines whether a drag partition is necessary, which in turn determines whether $\tau_b$ or $\tau_bs$ drives transport.

We use two tools:
1. the **Boguchwal-SOuthard depth-velocity-size diagram** which is visually interpretive
2. the **van den Berg & ban Gelder** method which is quantitative and valid at field depths

## Step 1a : Temperature Standardization
The Boguchwal-Southard diagram is compiled from flume data at 10$^\degree$C. Our Cedar Creek flow is at 20$^\degree$C, so we must convert grain size and velocity to their 10$^\degree$C equivalents before plotting:
$$ D_{10} = D_{20}* (\frac{\nu_{20}}{\nu_{10}})^{2/3} $$
$$ U_{10} = U_{20}* (\frac{\nu_{20}}{\nu_{10}})^{1/3} $$

From Southard Ch 12.

In [ ]:
# Temperature standardization
nu_ratio = nu / nu10                     # ν_20 / ν_10 ≈ 0.763
D50_10   = D50 * 1000 * nu_ratio**(2/3) # standardized D₅₀ in mm
U_10_bf  = U_bf * nu_ratio**(1/3)       # standardized U at bankfull

print(f'ν ratio (20°C / 10°C)  = {nu_ratio:.4f}')
print(f'D₅₀ at 20°C            = {D50*1000:.2f} mm')
print(f'D₅₀ standardized (10°C)= {D50_10:.3f} mm  (shifts left on diagram)')
print(f'U at 20°C              = {U_bf:.3f} m/s')
print(f'U standardized (10°C)  = {U_10_bf:.3f} m/s  (shifts down on diagram)')

## Step 1b : Plot on Boguchwal-Southard Diagram
**Depth caveat — read before plotting.** The diagram panel we are using is calibrated for flume depths of 0.25–0.40 m. Cedar Creek at bankfull has H ≈ 2.57 m. As depth increases, the dune field expands and the dune-to-upper-plane boundary shifts to higher velocities. Your point will likely plot above the upper-plane boundary of the flume panel — but this is the diagram telling you it cannot be directly applied, not that the flow is actually in upper plane bed. This is exactly why we follow up with the van den Berg & van Gelder equations in Step 1c.

#### Let's plot the Boguchwal-Southard diagram

In [ ]:
# Digitized from Southard & Boguchwal (1990) as reproduced in Southard Ch.12
bs = {
  'no_mvt':    ([0.08,0.10,0.15,0.20,0.30,0.50,1.0,2.0],
                [0.20,0.23,0.28,0.32,0.38,0.44,0.52,0.68]),
  'rip_dune':  ([0.08,0.10,0.15,0.20,0.25,0.40],
                [0.36,0.42,0.54,0.58,0.60,0.58]),
  'dune_up':   ([0.08,0.10,0.15,0.20,0.30,0.50,1.0,2.0],
                [0.60,0.68,0.78,0.84,0.90,0.95,1.00,1.08]),
  'Fr084':     ([0.08,2.0],[1.35,1.35]),
  'Fr100':     ([0.08,2.0],[1.60,1.60]),
}
labels = {'no_mvt':'Incipient motion','rip_dune':'Ripple → Dune',
          'dune_up':'Dune → Upper plane','Fr084':'Fr = 0.84','Fr100':'Fr = 1.0'}
styles = {'no_mvt':'k-','rip_dune':'b-','dune_up':'r-','Fr084':'k--','Fr100':'k:'}

fig, ax = plt.subplots(figsize=(10,8))
for key,(D,U) in bs.items():
    ax.semilogx(D, U, styles[key], lw=1.5, label=labels[key])

# Plot unstandardized point (open) and standardized point (filled)
ax.semilogx(D50*1000, U_bf, 'o', ms=10, mfc='none', mec='darkorange',
            mew=2, label=f'Cedar Ck, 20°C  ({D50*1000:.2f} mm, {U_bf:.2f} m/s)')
ax.semilogx(D50_10, U_10_bf, 's', ms=10, color='darkorange',
            label=f'Standardized to 10°C  ({D50_10:.3f} mm, {U_10_bf:.2f} m/s)')
ax.annotate('←  standardization\n    shifts point', xy=(D50_10, U_10_bf),
            xytext=(0.30, 1.20), fontsize=8, color='darkorange',
            arrowprops=dict(arrowstyle='->', color='darkorange', lw=1))

# Region labels
ax.text(0.15,0.50,'RIPPLES',fontsize=9,ha='left',color='navy')
ax.text(0.30,0.80,'DUNES',fontsize=10,ha='center',color='red')
ax.text(0.09,1.10,'UPPER PLANE',fontsize=9,ha='left',color='gray')
ax.text(0.15,0.24,'NO MOVEMENT',fontsize=9,ha='left',color='black')

ax.set_xlabel('Median grain size  D₅₀  (mm, standardized to 10°C)')
ax.set_ylabel('Mean flow velocity  U  (m/s, standardized to 10°C)')
ax.set_title('Boguchwal-Southard diagram  (d = 0.25–0.40 m panel)')
ax.set_xlim(0.07, 2.5); ax.set_ylim(0.15, 2.0)
ax.legend(fontsize=8,loc='upper left'); ax.grid(True,which='both',alpha=0.2)
# ax.text(0.55,0.18,'⚠ Flume depths only — field point may plot above boundaries',
#         transform=ax.transAxes,fontsize=8,color='red',ha='center')
plt.tight_layout()
plt.show()

### Questions Part 1:
#### Where does the standardized point plot? Does it sit int eh dune field, or has it crossed into upper plane bed? If it crossed: is that becasue the flow really is in upper plan bed, or because of the depth limitation of this diagram panel? THis is why Step 1c matters.

## Part 1c : van den Berg & van Gelder Classification
The van den Berg & van Gelder (1993) method is valid at field deptsh becasue it uses the grain roughness Shields stress $\tau_bs$, which accounts for actual flow depth, rather than assumign flume-scale geometry. This is the quantitative check.
| Equation  | Parameter  |
|---|---|
| $D^* = D_{50} * (\frac{s_{sg}*g}{\nu^2})^{1/3} $   | grain Reynolds parameter  |
| $C_{bs} = 18*log_{10}(12\frac{H}{D_{90}})$  | grain roughness Chézy coefficient |
| $\tau_{bs} = \frac{\rho*g*U^2}{C_{bs}^2} $  | grain roughness bed shear stress |
| $\tau_{bs}^* = \frac{\tau_{bs}}{(\rho_s-\rho)g*D_{50}} $  | grain roughness Shields stress |

**note that these are slightly different equations than are used when we do the Einstein partition later
#### Let's create and then impliment the van_den_Berg_classify() function.
##### First, let's create that function...

In [ ]:
def van_den_Berg_classify(D50, D90, H, U, rho_s=2650., rho=998.2, nu=1.00e-6, g=9.81):
    """
    Van den Berg & van Gelder (1993) bedform regime.
    Returns (theta_prime, D_star, regime_str).
    Approx. boundaries (see lecture Fig. 12-19 analog):
      θ′ < θ′_c(D*)          → no movement / lower plane
      θ′_c < θ′ < 0.4–0.6   → dunes  (boundary varies with D*)
      θ′ > ~0.55–0.80        → upper plane bed
    For D* > 10 (gravel), lower plane bed at all θ′ < 0.55.
    """
    s_sg   = (rho_s - rho) / rho
    D_star = D50 * ((s_sg * g) / nu**2) ** (1/3)
    C_prim = 18.0 * np.log10(_____)      
    tau_prim   = rho * g * U**2 / _____
    theta_prim = tau_prim / ((rho_s - rho) * g * D50)
    if theta_prim > 0.55:  regime = 'Upper plane bed'
    elif theta_prim < 0.06: regime = 'No movement / lower plane'
    else:                   regime = 'Dunes'
    return theta_prim, D_star, regime

##### ...annnnd now impliment it for Cedar Creek at bankfull flow.

In [ ]:
theta_p, D_star, regime = van_den_Berg_classify(D50, D90, H_bf, U_bf)
print(f'D*           = {D_star:.1f}')
print(f'$\tau_{bs}^*$          = {theta_p:.3f}')
print(f'Bedform regime: {regime}')
print()

### Questions Part 1c
#### The van den Berg & van Gelder method gives us $\tau_{bs}^*=0.47ish$, which is comfortably in the xune zone. The B-S diagram (flume depths) suggested upper plane bed. What phyiscal variable does vdBvG account for that the B-S diagram cannot? Why does this matter for what we do next?

# Part 2 -- the Einstein Drag Partition
Dunes are confirmed via the above vdBvG calculations. This matters because dune form drag is large, but does not move sediment, only the skin friction $\tau_bs$ acts on grains. The Einstein (1950) method finds the depth $H_s$ that would produce the observed velocity U over a *flat* bed with grain roughness only. The rest of $\tau_b$ will be form drag.

We will solve iteratively for $H_s$ using:

$$ Total $$
| parameter  | equation  |
|---| --- |
| total bed shear stress | $ \tau_b = \tau_{bs} + \tau{bf} $  |
| total coefficient of friction  | $ C_f = C_{fs} + C_{ff} $ |
| total depth  | $ H = H_s + H_f $ |

$$ Individual $$
| parameter  | equation  |
|---| --- |
| total bed shear stress | $ \tau_b = \rho g H S = \rho C_f U^2 $  |
| skin friction bed shear stress  | $ \tau_{bs} = \rho g H_s S = \rho C_{fs} U^2 $ |
| form drag bed shear stress  | $ \tau_{bf} = \rho g H_f S = \rho C_{ff} U^2 $ |
| skin friction coefficient of friction | $ C_{fs} = [\frac{1}{\kappa}*ln(11 \frac{H_s}{k_s})]^{-2} $ | 
| skin friction depth | $ H_s = \frac{U^2\kappa^2}{gS}*[ln(11 \frac{H_s}{k_s})]^{-2}$ |

$$ U = \frac{1}{\kappa}*\sqrt{(g*H_s*S)* ln(11*\frac{H_s}{k_s})} $$



In [ ]:
def iterate(H_s, U, k_s, kappa=0.41, tolerance = 1e-4, max_iterations=1000, verbose = False):
    if verbose:
        print(f'-----Beginning Iteration to determine H_s ---------------')
    coeffs = (U**2 * kappa**2)/(g*S)

    Hs_guess = H_s
    for iteration in range(max_iterations):
        Hs_ks = Hs_guess/k_s
        ln_value = np.log(11*Hs_ks)
        ln_2 = ln_value**(2)
        Hs_new = coeffs/ln_2
        if verbose:
            print(f'     guess H_s = {Hs_guess} | gives us new H_s = {Hs_new}')

        # check for convergence
        if np.isclose(Hs_new, Hs_guess, rtol=tolerance):
            if verbose:
                print(f'-----H_s = {Hs_new:.2f} m----------------------------------------')
            return Hs_new

        # reassign for next iteration
        Hs_guess = Hs_new
    # if max iterations reached
    print(f"Warning: Did not converge after {max_iterations} iterations")
    return Hs_guess

In [ ]:
def einstein_partition(U, H, S, k_s, rho=998.2, g=9.81, kappa=0.41, verbose=False):
    """
    Einstein (1950) drag partition.

    Returns
    -------
    H_s   : skin-friction flow depth [m]
    tau_bs: skin-friction shear stress [Pa]
    tau_bf: form-drag shear stress [Pa]
    """
    # calculate total bed shear stress
    tau_b = ____________________
    #print(f"total bed shear stress: {tau_b}")

    # calculate total coefficient of friction
    C_f = tau_b/(rho*(U**2))
    #print(f"total coefficient of friction: {C_f}")
    
   # iterate to figure out H_S
    H_s = iterate(H, U, k_s, verbose=verbose)
    #print(f"skin friction depth: {H_s}")

    # calculate skin friction coefficient of friction
    const = 1/kappa
    Hs_ks = H_s/k_s
    ln_value = np.log(11*Hs_ks)
    C_fs = (const*ln_value)**(-2)
    #print(f"skin friction coefficient of friction: {C_fs}")

    # calculate skin friction bed shear stress
    tau_bs = _______________
    #print(f"skin friction bed shear stress: {tau_bs}")

    # calculate form drag bed shear stress
    tau_bf = ____ - _____

    # caluculate form drag coefficient of friction
    C_ff = ____ - _____

    # calculate form drag depth
    H_f = H-H_s
    
    return tau_b, tau_bs, tau_bf, H_s

In [ ]:
tau_b, tau_bs, tau_bf, H_s = einstein_partition(U_bf, H_bf, S, k_s, verbose=True)
tau_b = rho * g * H_bf * S

print(f'H (total depth)        = {H_bf:.2f} m')
print(f'H_s (skin-friction depth)= {H_s:.2f} m  ({100*H_s/H_bf:.0f}% of H)')
print(f'τ_b  (total)           = {tau_b:.2f} Pa')
print(f'τ_bs (skin friction)   = {tau_bs:.2f} Pa  ({100*tau_bs/tau_b:.0f}% of τ_b)')
print(f'τ_bf (form drag)       = {tau_bf:.2f} Pa  ({100*tau_bf/tau_b:.0f}% of τ_b)')


In [ ]:
# ── Fig 2a: Partition bar chart at bankfull ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
bars = ax.bar(['τ_b\n(total)', 'τ_bs\n(skin)', 'τ_bf\n(form)'],
              [tau_b, tau_bs, tau_bf],
              color=['steelblue','darkorange','lightgray'], width=0.5, edgecolor='k', lw=0.5)
ax.set_ylabel('Shear stress  (Pa)')
ax.set_title('Drag partition at bankfull  (Q = 250 m³/s)')
for bar, pct in zip(bars[1:], [tau_bs/tau_b, tau_bf/tau_b]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()/2,
            f'{100*pct:.0f}%', ha='center', va='center',
            fontsize=12, fontweight='bold', color='white' if pct>0.3 else 'gray')
ax.set_ylim(0, tau_b * 1.2)

# ── Fig 2b: H_s/H fraction across all discharges ─────────────────
ax = axes[1]
Hs_vals  = np.array([einstein_partition(U_vals[i], H_vals[i], S, k_s)[0] for i in range(len(Q_vals))])
frac_bs  = Hs_vals / H_vals
frac_bf  = 1.0 - frac_bs

ax.plot(Q_vals, frac_bs * 100, 'o-', color='darkorange', lw=2, label='τ_bs  (skin friction)')
ax.plot(Q_vals, frac_bf * 100, 's--', color='lightcoral', lw=2, label='τ_bf  (form drag)')
ax.axvline(Q_bf, color='gray', ls=':', lw=1.5, label='Bankfull Q')
ax.set_xlabel('Discharge  Q  (m³/s)')
ax.set_ylabel('Fraction of τ_b  (%)')
ax.set_title('Partition fraction across flow range')
ax.set_ylim(0, 100); ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle('Cedar Creek — Einstein Drag Partition', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout(); 
plt.show()

### Questions Part 2
#### 60% of bed shear stress is form drag around dune lee faces. This stress does NOT move sediment. If you applied $\tau_b$ directly to MPM, by what factor would you overestimate the transport forcing? Keep this number in mind for Part 3.

# Part 3 -- MPM Bedload Transport
Now we're going to apply the Wong & Parker corrected MPM formula using $\tau_{bs}^*$, the Shields stress computed from $\tau_{bs}$, not $\tau_b$:
$$ \tau_{bs}^* = \frac{\tau_{bs}}{(\rho_s - \rho)gD_{50}} $$

<p><center>[Wong & Parker 2006, correcting Meyer-Peter & Müller 1948]</center>
\begin{equation}
q_b^* = 3.97\left(\tau_{bs}^* - 0.0495\right)^{1.5}
\end{equation}
<p><center>[ASCE Eq. 2-90a]</center>
\begin{equation}
q_b = q_b^* \cdot \sqrt{(s-1)\, g\, D_{50}^3}
\end{equation}

In [ ]:
def compute_shields(tau, D, rho_s=2650., rho=998.2, g=9.81):
    """Dimensionless Shields stress from a shear stress [Pa] and grain size [m]."""
    return tau / ((rho_s - rho)*g*D)

def compute_MPM_bedload(tau_star, tau_star_c, D, rho_s=2650., rho=998.2, g=9.81):
    """
    Wong-Parker corrected MPM.  Returns q_b [m²/s].
    Zero where tau_star <= tau_star_c.
    """
    R      = (rho_s - rho) / rho
    excess = np.maximum(tau_star - tau_star_c, 0.0)
    q_b_star = 3.97 * excess ** _______   
    return q_b_star * np.sqrt(R * g * ______**3)  


#### Corrected vs uncorrected bedload at bankful

In [ ]:
tau_b_bf = rho * g * H_bf * S

tau_star_bs    = compute_shields(tau_bs, D50)   # uses τ_bs
tau_star_b    = compute_shields(tau_b, D50) # uses τ_b  (WRONG)
q_b_corr   = compute_MPM_bedload(tau_star_bs, tau_star_c, D50)
q_b_uncorr = compute_MPM_bedload(tau_star_b, tau_star_c, D50)

print(f'θ*_s  (skin friction)  = {tau_star_bs:.7f}')
print(f'θ*    (total stress)   = {tau_star_b:.3f}  ← this is wrong for dune bed')
print(f'q_b  (corrected)       = {q_b_corr:.2e} m²/s')
print(f'q_b  (uncorrected)     = {q_b_uncorr:.2e} m²/s')
print(f'Overestimate factor    = {q_b_uncorr/q_b_corr:.1f}×')



In [ ]:
# ── Fig 3: Bedload rating curves — corrected vs. uncorrected ─────
# Unpack all four return values from your einstein_partition
results    = [einstein_partition(U_vals[i], H_vals[i], S, k_s) for i in range(len(Q_vals))]
tau_b_all  = np.array([r[0] for r in results])   # total stress
tau_bs_all = np.array([r[1] for r in results])   # skin friction only

theta_s_all    = compute_shields(tau_bs_all, D50)   # corrected
theta_b_all    = compute_shields(tau_b_all,  D50)   # uncorrected (WRONG for dune bed)

q_b_corr_all   = np.array([compute_MPM_bedload(th, tau_star_c, D50) for th in theta_s_all])
q_b_uncorr_all = np.array([compute_MPM_bedload(th, tau_star_c, D50) for th in theta_b_all])

# Use np.where instead of boolean masking — more robust
idx_c = np.where(q_b_corr_all   > 0)[0]
idx_u = np.where(q_b_uncorr_all > 0)[0]
idx   = np.intersect1d(idx_c, idx_u)   # discharges where both are above threshold

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: log-scale rating curves
ax = axes[0]
ax.semilogy(Q_vals[idx_c], q_b_corr_all[idx_c],   'o-', color='darkorange',
            lw=2, label='q_b corrected  (uses τ_bs)')
ax.semilogy(Q_vals[idx_u], q_b_uncorr_all[idx_u], 's--', color='steelblue',
            lw=2, label='q_b uncorrected  (uses τ_b, WRONG)')
ax.axvline(Q_bf, color='gray', ls=':', lw=1.5, label='Bankfull Q')
ax.set_xlabel('Q  (m³/s)'); ax.set_ylabel('q_b  (m²/s)')
ax.set_title('Bedload rating curves — Cedar Creek')
ax.legend(fontsize=9); ax.grid(True, which='both', alpha=0.3)

# Right: overestimate factor vs Q
ax = axes[1]
factor = q_b_uncorr_all[idx] / q_b_corr_all[idx]
ax.plot(Q_vals[idx], factor, 'D-', color='crimson', lw=2)
ax.axhline(1.0, color='gray', ls='--', lw=1)
ax.axvline(Q_bf, color='gray', ls=':', lw=1.5, label='Bankfull Q')
ax.set_xlabel('Q  (m³/s)'); ax.set_ylabel('q_b uncorrected / q_b corrected')
ax.set_title('Overestimate factor vs. discharge')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.text(0.05, 0.9, 'Factor grows nonlinearly\nwith discharge',
        transform=ax.transAxes, fontsize=9, color='crimson')

plt.suptitle('MPM bedload: the cost of skipping the drag partition', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

# Part 4 -- Settling Velocity
The Ferguson & Church (2004) formula covers the full range from Stokes (fine silt, viscous drag) to fully turbulent (coarse sand), with a smooth transition between them:
$$ \omega_s = \frac{RgD^2}{C_1\nu + \sqrt{0.75C_2RgD^3}} $$
$$ C_1 = 18 \space(Stokes), \space C_2 = 1.0 \space(Impact\space Law), \space R = \frac{\rho_s-\rho}{\rho} $$

In [ ]:
def settling_velocity(D_m, rho_s=2650., rho=998.2, nu=1.00e-6, g=9.81, C1=18.0, C2=1.0):
    """
    Ferguson & Church (2004) settling velocity.
    D_m : grain diameter [m].  Returns w_s [m/s].
    """
    R   = (rho_s - rho) / rho
    num = R * g * D_m**2
    denom = C1 * nu + np.sqrt(0.75 * C2 * R * g * _______**3)  
    return num / denom

In [ ]:
D_range = np.logspace(-5, -1, 400)     # 0.01 mm → 100 mm
ws_range = settling_velocity(D_range)

fig, ax = plt.subplots(figsize=(7,4.5))
ax.loglog(D_range*1000, ws_range*100, 'k-', lw=2)
ax.axvline(D50*1000, color='darkorange', ls='--', lw=2,
           label=f'Cedar Creek D₅₀ = {D50*1000:.1f} mm')
ax.axvline(0.0625, color='steelblue', ls=':', lw=1.5, label='Sand/silt boundary (0.0625 mm)')
ax.axvline(2.0,    color='saddlebrown', ls=':', lw=1.5, label='Sand/gravel boundary (2 mm)')
ax.scatter([D50*1000],[settling_velocity(D50)*100], s=80, color='darkorange', zorder=5)

ws_D50 = settling_velocity(D50)
ax.annotate(f'w_s = {ws_D50*100:.2f} cm/s',
            xy=(D50*1000, ws_D50*100), xytext=(0.5, 0.3),
            textcoords='axes fraction', fontsize=9, color='darkorange',
            arrowprops=dict(arrowstyle='->', color='darkorange'))
ax.set_xlabel('Grain diameter  D  (mm)'); ax.set_ylabel('w_s  (cm/s)')
ax.set_title('Ferguson-Church (2004) settling velocity')
ax.legend(fontsize=9); ax.grid(True,which='both',alpha=0.3)
plt.tight_layout(); plt.show()
print(f'w_s at D₅₀ = {D50*1000:.1f} mm:  {ws_D50*100:.2f} cm/s')


# Part 5 -- Rouse number and Concentration Profile
The **Rouse number Z** sets the shape of the suspension profile. Crucially, it uses $u_{*s} = \sqrt{\frac{\tau_{bs}}{\rho}}$ (the skin friction shear velocity), not f$u_*$. Form drag turbulence is generated in the lee of dunes but does not diffuse sediment upward in the same way:
| Equation  | Parameter  |
|---|---|
| $u_{*s} = \sqrt{\frac{\tau_{bs}}{\rho}} $   | skin friction shear velocity  |
| $Z = \frac{\omega_s}{\kappa u_{*s}}$  | Rouse number |
| $ \frac{C(z)}{C_a} = [\frac{H-h}{h}*\frac{h_a}{H-h_a}]^Z $  | Rouse-Vanonie-Ippen profile |

<p>The suspension threshold is <b> Z < 5.4 </b> (i.e., absolutely no suspended sediment above 5.4)</p>
<p>moderate suspension is <b>1.2 < Z < 2.5 </b> </p>
<p>Strong suspension is <b>Z < 1.2 </b></p>


#### Let's compute the Rouse number using the total bed shear stress and skin friction bed shear stress

In [ ]:
ws_D50   = settling_velocity(D50)
u_star_s = np.sqrt(tau_bs / rho)       # correct: skin-friction only
u_star_t = np.sqrt(tau_b / rho)     # wrong: total stress

Z_correct = ws_D50 / (kappa * u_star_s)
Z_wrong   = ws_D50 / (kappa * u_star_t)

print(f'u*_s  (skin friction)  = {u_star_s:.4f} m/s')
print(f'u*    (total stress)   = {u_star_t:.4f} m/s  (higher — inflated by form drag)')
print(f'Z correct              = {Z_correct:.2f}  →  {"strong" if Z_correct<1.2 else "moderate" if Z_correct<2.5 else "weak"} suspension')
print(f'Z wrong                = {Z_wrong:.2f}  →  {"strong" if Z_wrong<1.2 else "moderate" if Z_wrong<2.5 else "weak"} suspension')
print()
print(f'Using total u* instead of u*_s predicts a {"more" if Z_wrong<Z_correct else "less"} uniform profile')
print(f'and misclassifies the regime: Z crosses the Z=1.2 boundary.')



#### Now let's compute the Rose profile for each of these Rouse numbers

In [ ]:
def rouse_profile(Z, H, h_a=None, n=200):
    if h_a is None: h_a = 0.05 * H
    h = np.linspace(h_a + 1e-4, H * 0.999, n)
    C_rel = ((H - h) / h * h_a / (H - h_a)) ** Z
    return h, C_rel / C_rel.max()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5.5))
for Z_val, col, lbl in [
    (Z_correct,'darkorange',f'Z = {Z_correct:.2f}  (correct, u*_s)'),
    (Z_wrong,  'steelblue', f'Z = {Z_wrong:.2f}  (wrong, u*_total)')]:
    h, C = rouse_profile(Z_val, H_bf)
    ax.plot(C, h/H_bf, lw=2.5, color=col, label=lbl)

ax.axhline(0.05, color='gray', ls=':', lw=1, label='Reference level z_a = 0.05H')
ax.axvline(0, color='k', lw=0.5)
ax.set_xscale('log')
ax.set_xlim(1e-3, 1.05)
ax.set_xlabel('Relative concentration  C(z)/C_max  (log scale)')
ax.set_ylabel('Relative depth  z/H')
ax.set_title(f'Rouse profiles — Cedar Creek bankfull\nH = {H_bf:.2f} m, D₅₀ = {D50*1000:.1f} mm')
ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_xlim(0, 1.05); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()


### Questions Part 5
#### The two profiles predict meaningfully different concentration distributions. Which profile is steeper (more sediment near the bed), and why? If you were sampling suspended sediment at mid-depth in Cedar Creek, would using the wrong Z cause you to over- or under-estimate the depth-averaged concentrations?

# Part 6 -- Translating to suspended load flux
Let's calculate the depth-average suspended load flux (using a provided $C_a$)

In [ ]:
# C_a = near-bed reference concentration [kg/m³]
# Estimated from Garcia-Parker (1991) for these conditions — provided.
C_a = 0.012   # kg/m³  ← ~12 mg/L, typical dune-bed sand at bankfull

# Numerical integration of C(z)·u(z) over the water column
# u(z) = (u*_s / κ) · ln(z / z_0)   where z_0 = k_s / 30
h_0  = k_s / 30.0
h_a  = 0.05 * H_bf
h_int = np.linspace(h_a + 1e-4, H_bf * 0.999, 500)

C_profile = C_a * ((H_bf - h_int)/h_int * h_a/(H_bf - h_a))**Z_correct
u_profile = (u_star_s / kappa) * np.log(h_int / h_0)
u_profile = np.maximum(u_profile, 0.0)   # clamp below z_0

q_s = np.trapezoid(C_profile * u_profile, h_int) / rho_s  # m²/s (volumetric)

print(f'Bedload transport rate    q_b = {q_b_corr:.2e} m²/s')
print(f'Suspended load flux       q_s = {q_s:.2e} m²/s')
print(f'Ratio  q_s / q_b              = {q_s/q_b_corr:.1f}')
print(f'Total bed-material load   q_t = {q_b_corr + q_s:.2e} m²/s')
print(f'Suspended fraction            = {100*q_s/(q_b_corr+q_s):.0f}%')

In [ ]:
# ── Fig 5b: Three-panel C(z), u(z), and C(z)×u(z) ──────────────
# This figure motivates the suspended load integration directly.
h_a   = 0.05 * H_bf
h_int = np.linspace(h_a + 1e-4, H_bf * 0.999, 500)

# Concentration profile (absolute, using reference C_a)
C_a   = 0.012             # kg/m³  reference concentration (Garcia-Parker est.)
C_abs = C_a * ((H_bf - h_int)/h_int * h_a/(H_bf - h_a))**Z_correct

# Log-velocity profile  u(h) = (u*_s/κ) ln(h/h_0), h_0 = k_s/30
h_0   = k_s / 30.0
u_log = (u_star_s / kappa) * np.log(h_int / h_0)
u_log = np.maximum(u_log, 0.0)   # clamp below h_0

# Local suspended flux = C(z) × u(z)  [kg/m²/s]
flux  = C_abs * u_log

fig, axes = plt.subplots(1, 3, figsize=(12, 5.5), sharey=True)

# Panel 1: Concentration
ax = axes[0]
ax.semilogx(C_abs * 1000, h_int / H_bf, lw=2.5, color='darkorange')
ax.axhline(0.05, color='gray', ls=':', lw=1, label='h_a = 0.05H')
ax.set_xlabel('C(z)  (mg/L)')
ax.set_ylabel('z / H')
ax.set_title('Concentration  C(z)\n(Rouse profile, log scale)')
ax.legend(fontsize=8); ax.grid(True, which='both', alpha=0.3)
ax.set_ylim(0, 1)

# Panel 2: Velocity
ax = axes[1]
ax.plot(u_log, h_int / H_bf, lw=2.5, color='steelblue')
ax.axhline(0.05, color='gray', ls=':', lw=1)
ax.set_xlabel('u(z)  (m/s)')
ax.set_title('Velocity  u(z)\n(log profile, u*_s)')
ax.grid(alpha=0.3)

# Panel 3: Local suspended flux
ax = axes[2]
ax.plot(flux * 1000, h_int / H_bf, lw=2.5, color='purple')
ax.axhline(0.05, color='gray', ls=':', lw=1)
peak_idx = np.argmax(flux)
ax.axhline(h_int[peak_idx]/H_bf, color='purple', ls='--', lw=1,
           label=f'Peak flux at z/H = {h_int[peak_idx]/H_bf:.2f}')
ax.set_xlabel('C(z)·u(z)  (mg/m²/s)')
ax.set_title('Local suspended flux  C·u\n(integrand for q_s)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Integrate to get q_s
q_s = np.trapezoid(C_abs * u_log, h_int) / rho_s
q_b_corr = compute_MPM_bedload(compute_shields(tau_bs, D50), tau_star_c, D50)

plt.suptitle(f'Cedar Creek bankfull — suspended load structure\n'
             f'q_s ≈ {q_s:.2e} m²/s   |   q_b ≈ {q_b_corr:.2e} m²/s   |   q_s/q_b ≈ {q_s/q_b_corr:.1f}',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'\nTotal bed-material load  q_t = q_b + q_s = {(q_b_corr+q_s):.2e} m²/s')
print(f'Suspended fraction             = {100*q_s/(q_b_corr+q_s):.0f}%')
print(f'Peak suspended flux at z/H     = {h_int[peak_idx]/H_bf:.2f}')


# Part 7 -- Synthesis
Answer the following questions

1. **Which mode dominates, bedload or suspended load?** *At bankfull, what fraction of the total bed-material load is carried in suspension versus bedload? Does this match your expectation given Z=1.78? Recall Southard: at what Rouse number does suspended load typically begin to exceed bedload?*
2. **What if you had skipped the partition?** *You computed $q_b$ using $\tau_{bs}$ and $Z$ using $u_{*s}$. Recompute $q_{bs}$ using `Z_wrong` and compare the  resulting $\frac{q_{bs}}{q_b}$ ratio to the corrected version. In which direction does the error go? Does skipping hte partition make suspended load look artificially more or less dominant? Explain why in terms of what `Z_wrong` does to the concentration profile shape.
<p>--------------------<b>ANSWER AFTER YOU COMPLETE THE WEEKEND HW</b>----------------</p>
3. <b>Connecting back to the weekend Scenarios.</b> In Scenario A, Z dropped to ~0.52. Without doing the calculation, would you expect $\frac{q_{bs}}{q_b}$ to be larger or smaller than the base case, and by roughly what order of magnitude? What physical changes (in $\omega_s$, in the concentration profile shape, in bedload threshold) all push in the same direction?*


---------------------------------------------
# Over the weekend...
Now that you have a feel for how to use these functions, let's apply them to a reach of the Cumberland River near Nashbille. Your job is to read the figures, answer teh questions, and then **change one parameter at a time** in each scenario cell and re-examine the output. The goal is physical intuition: what happens and why?

Workflow:
1. Run all cells, in order, for the base case.
2. Answer Q1-4.
3. Work through Scenarios A, B, and C (each changes exactly one parameter).
    - for each scenario, re-run the analysis cells and ansswwer the scenario question.
  
**You do not need to write new functions!**

# Base Case -- The Cumberland Reach
### ─── ALL PARAMETERS LIVE HERE ────────────────────────────────────
#### To run a scenario: change ONLY the parameter marked for that scenario, then re-run all cells below this one.

In [ ]:
D50 = 0.25e-3      # m  ← Scenario A changes this
D90 = 0.45e-3      # m
S   = 0.00008      # ← Scenario B changes this
Q   = 1400.0       # m³/s  ← Scenario C changes this

# Fixed for all scenarios
W     = 160.0
n_man = 0.022
k_s   = 2.5 * D50

# Compute hydraulics from Manning
H = (Q * n_man / (W * S**0.5)) ** (3/5)
U = Q / (W * H)
tau_b = rho * g * H * S
print(f'H = {H:.2f} m   U = {U:.2f} m/s   τ_b = {tau_b:.2f} Pa')

# Step 1 : Bedform Regime

In [ ]:
# Temperature standardization
nu_ratio = nu / nu10
D50_10   = D50 * 1000 * nu_ratio**(2/3)
U_10     = U * nu_ratio**(1/3)

# vdBvG classification
theta_p, D_star, regime = van_den_Berg_classify(D50, D90, H, U)
print(f'Standardized: D₁₀ = {D50_10:.3f} mm,  U₁₀ = {U_10:.3f} m/s')
print(f'D*   = {D_star:.1f}     θ′ = {theta_p:.3f}')
print(f'Regime: {regime}')

# Boguchwal-Southard diagram (boundaries as in class)
fig, ax = plt.subplots(figsize=(10,8))
for key,(D,Uv) in bs.items():
    ax.semilogx(D, Uv, styles[key], lw=1.5, label=labels[key])
ax.semilogx(D50*1000, U, 'o', ms=10, mfc='none', mec='purple', mew=2,
            label='Cumberland Reach 20°C')
ax.semilogx(D50_10, U_10, 's', ms=10, color='purple',
            label=f'Standardized: ({D50_10:.3f} mm, {U_10:.3f} m/s)')

ax.text(0.15,0.50,'RIPPLES',fontsize=9,ha='left',color='navy')
ax.text(0.30,0.80,'DUNES',fontsize=10,ha='center',color='red')
ax.text(0.09,1.10,'UPPER PLANE',fontsize=9,ha='left',color='gray')
ax.text(0.15,0.24,'NO MOVEMENT',fontsize=9,ha='left',color='black')

ax.set_xlabel('D₅₀ (mm, standardized 10°C)'); ax.set_ylabel('U (m/s, standardized 10°C)')
ax.set_title(f'Boguchwal-Southard  |  VBvG: θ′={theta_p:.3f}  →  {regime}')
ax.set_xlim(0.07,2.5); ax.set_ylim(0.15,2.0)
ax.legend(fontsize=8,loc='upper left'); ax.grid(True,which='both',alpha=0.2)

plt.tight_layout(); 
plt.show()

In [ ]:
# ── Fig W0: Full hydraulics + partition rating curves ────────────
# Pre-compute across a discharge range so figures regenerate per scenario
Q_range  = np.linspace(50, Q * 1.5, 60)
H_range  = (Q_range * n_man / (W * S**0.5)) ** (3/5)
U_range  = Q_range / (W * H_range)
tb_range = rho * g * H_range * S
k_s_now  = 2.5 * D50

# Compute τ_bs at each discharge (may take a second)
tbs_range = np.array([einstein_partition(U_range[i], H_range[i], S, k_s_now)[1]
                      for i in range(len(Q_range))])

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# H vs Q
ax = axes[0,0]
ax.plot(Q_range, H_range, '-', color='steelblue', lw=2)
ax.axvline(Q, color='gray', ls=':', lw=1.5, label=f'Current Q = {Q:.0f} m³/s')
ax.set_xlabel('Q  (m³/s)'); ax.set_ylabel('H  (m)')
ax.set_title('Flow depth'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# U vs Q
ax = axes[0,1]
ax.plot(Q_range, U_range, '-', color='teal', lw=2)
ax.axvline(Q, color='gray', ls=':', lw=1.5)
ax.set_xlabel('Q  (m³/s)'); ax.set_ylabel('U  (m/s)')
ax.set_title('Mean velocity'); ax.grid(alpha=0.3)

# τ_b and τ_bs vs Q
ax = axes[1,0]
ax.plot(Q_range, tb_range,  '-',  color='navy',       lw=2, label='τ_b  (total)')
ax.plot(Q_range, tbs_range, '--', color='darkorange',  lw=2, label='τ_bs (skin friction)')
ax.axvline(Q, color='gray', ls=':', lw=1.5)
ax.fill_between(Q_range, tbs_range, tb_range, alpha=0.15, color='lightcoral',
                label='Form drag τ_bf')
ax.set_xlabel('Q  (m³/s)'); ax.set_ylabel('Stress  (Pa)')
ax.set_title('Drag partition across flow range'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# τ_bs/τ_b fraction vs Q
ax = axes[1,1]
frac = tbs_range / tb_range
ax.plot(Q_range, frac * 100, '-', color='darkorange', lw=2)
ax.axvline(Q, color='gray', ls=':', lw=1.5, label=f'Current Q')
ax.axhline(50, color='gray', ls='--', lw=1, alpha=0.5)
ax.set_xlabel('Q  (m³/s)'); ax.set_ylabel('τ_bs / τ_b  (%)')
ax.set_title('Skin-friction fraction of total stress')
ax.set_ylim(0, 100); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle(f'Cumberland Reach — Hydraulics & Partition  |  D₅₀={D50*1000:.2f}mm  S={S:.5f}  W={W:.0f}m',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()


## Q1
Where does the Cumberland Reach plot on the B-S diagram? What does the vdBvG give for $\tau_{bs}^*$ and D*? Is this a dune bed? Does the answer change between the flume diagram and the vdBvG equations?--and if so, which should you trust for a 6 m deep river?

# Step 2 : Einstein Drag Partition

In [ ]:
tau_b_val, tau_bs, tau_bf, H_s = einstein_partition(U, H, S, k_s)

print(f'τ_b   = {tau_b_val:.2f} Pa')
print(f'τ_bs  = {tau_bs:.2f} Pa  ({100*tau_bs/tau_b_val:.0f}% skin friction)')
print(f'τ_bf  = {tau_bf:.2f} Pa  ({100*tau_bf/tau_b_val:.0f}% form drag)')

# Bar chart
fig, ax = plt.subplots(figsize=(5,3.5))
ax.bar(['τ_b\n(total)'],[tau_b_val],color='steelblue',width=0.4,label='Total')
ax.bar(['τ_bs\n(skin)','τ_bf\n(form)'],[tau_bs,tau_bf],
       color=['darkorange','lightgray'],width=0.4)

ax.set_ylabel('Shear stress (Pa)'); ax.set_title('Drag partition — Cumberland Reach')
ax.text(1,tau_bs/2,f'{100*tau_bs/tau_b_val:.0f}%',ha='center',va='center',fontsize=11,color='white',fontweight='bold')
ax.text(2,tau_bs+tau_bf/2,f'{100*tau_bf/tau_b_val:.0f}%',ha='center',va='center',fontsize=11,color='gray',fontweight='bold')

plt.tight_layout(); 
plt.show()

## Q2
What fraction of $\tau_b$ is doing actual transport work? What fraction is being dissipated as form drag around dune lee faces? If you used $\tau_b$ directly in your transport calculation, would you over- or under-estimate $q_b$ and by roughly what factor?

# Step 3 : Bedload Transport

In [ ]:
tau_star_bs  = compute_shields(tau_bs, D50)
tau_star_b  = compute_shields(tau_b_val, D50)
q_b_c    = compute_MPM_bedload(tau_star_bs, tau_star_c, D50)
q_b_u    = compute_MPM_bedload(tau_star_b, tau_star_c, D50)


print(f'θ*_s  = {tau_star_bs:.3f}  (corrected)')
print(f'θ*    = {tau_star_b:.3f}  (uncorrected)')
print(f'q_b corrected   = {q_b_c:.2e} m²/s')
print(f'q_b uncorrected = {q_b_u:.2e} m²/s')
print(f'Overestimate factor = {q_b_u/q_b_c:.1f}×  (if you skip the partition)')


## Q3
By what factor does skipping the partion overestimate bedload for the Cumberland Reach base case? Compare this to Cedar Creek from class. WHich river has a larger form-drag fraction, and why might that be?

# Step 4 : Settling Velocity and Rouse Profile

In [ ]:
ws    = settling_velocity(D50)
u_s_s = np.sqrt(tau_bs / rho)
Z     = ws / (kappa * u_s_s)

print(f'w_s (D₅₀ = {D50*1000:.2f} mm)  = {ws*100:.2f} cm/s')
print(f'u*_s                    = {u_s_s:.4f} m/s')
print(f'Z (Rouse number)        = {Z:.2f}')
if   Z < 1.2:  susp = 'Strong suspension'
elif Z < 2.5:  susp = 'Moderate suspension'
elif Z < 5.4:  susp = 'Incipient / weak suspension'
else:           susp = 'No suspension'
print(f'Regime: {susp}')

z_vals, C_rel = rouse_profile(Z, H)

fig, ax = plt.subplots(figsize=(5,5))

ax.plot(C_rel, z_vals/H, lw=2.5, color='purple',
        label=f'Z = {Z:.2f}  ({susp})')
ax.axhline(0.05, color='gray', ls=':', lw=1, label='z_a = 0.05H')

ax.set_xlabel('Relative concentration  C(z)/C_a  (log scale)')
ax.set_ylabel('Relative depth  z/H')
ax.set_title(f'Rouse profile — Cumberland Reach\nD₅₀={D50*1000:.2f} mm, Z={Z:.2f}')
ax.set_xscale('log'); ax.set_xlim(0.01, 1.5); ax.set_ylim(0, 1)
ax.legend(fontsize=9); ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()


## Q4
Describe the shape of the Rouse profile for the base Cumberland Reach. Is sediment distributed relatively uniformly in the water column, or is it concentrated near teh bed? How does this compare to Cedar Creek (Z=1.78)?

In [ ]:
# ── Fig W4b: Z vs Q across the full discharge range ─────────────
ws_now = settling_velocity(D50)
Z_range = ws_now / (kappa * np.sqrt(tbs_range / rho))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogy(Q_range, Z_range, '-', color='purple', lw=2.5,
            label=f'Z  (D₅₀ = {D50*1000:.2f} mm, S = {S:.5f})')
ax.axvline(Q, color='gray', ls=':', lw=1.5, label=f'Current Q = {Q:.0f} m³/s')

# Threshold lines
ax.axhline(5.4, color='saddlebrown', ls='--', lw=1.5, label='Z = 5.4  no suspension')
ax.axhline(2.5, color='goldenrod',   ls='--', lw=1.5, label='Z = 2.5  incipient')
ax.axhline(1.2, color='steelblue',   ls='--', lw=1.5, label='Z = 1.2  strong suspension')

# Shaded regime bands
ax.fill_between(Q_range, 1.2, 2.5, alpha=0.07, color='goldenrod')
ax.fill_between(Q_range, 0.0, 1.2, alpha=0.07, color='steelblue')

# Annotate regime labels
ax.text(Q_range[-1]*0.98, 4.0,   'Incipient',      ha='right', fontsize=8, color='saddlebrown')
ax.text(Q_range[-1]*0.98, 1.8,   'Moderate',       ha='right', fontsize=8, color='goldenrod')
ax.text(Q_range[-1]*0.98, 0.7,   'Strong',         ha='right', fontsize=8, color='steelblue')

ax.set_xlabel('Discharge  Q  (m³/s)')
ax.set_ylabel('Rouse number  Z')
ax.set_title(f'Cumberland Reach — Z vs. Q  |  D₅₀={D50*1000:.2f}mm  S={S:.5f}')
ax.legend(fontsize=8, loc='upper right'); ax.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()

# Find discharge where suspension first becomes active
idx_thresh = np.where(Z_range < 5.4)[0]
if len(idx_thresh) > 0:
    print(f'Suspension first active near Q = {Q_range[idx_thresh[0]]:.0f} m³/s  (Z crosses 5.4)')
idx_strong = np.where(Z_range < 1.2)[0]
if len(idx_strong) > 0:
    print(f'Strong suspension begins near Q = {Q_range[idx_strong[0]]:.0f} m³/s  (Z crosses 1.2)')


# Scenarios : Change one variable at a time
For each scenario below:
1. go back to the first cell of this section and change ONE parameter as indicated
2. re-run all cells from there on
3. answer the scenario question here.

The table below shows the expected regime outcome **DO NOT LOOK UNTIL YOU HAVE MADE YOUR PREDICTION FIRST**

#### Table

| Scenario  | Parameter Change  | expected outcome
|---| --- | --- |
| A | $D_{50}$: 0.25 $\rightarrow$ 0.15 mm  |   <p> $\tau_{bs}^*$ jumps to ~0.81, crosses into upper plane bed; </p> $\tau_{bs} \approx \tau_b$ ; Z drops to ~0.52 |
| B | $S$: 0.00008 $\rightarrow$ 0.0003 | <p> $\tau_{bs}^*$ reaches ~1.24, also upper plane bed; </p> $\tau_{bs} \approx \tau_b$ ; Z $\approx$ 0.70  |
| C | $Q$: 1400 $\rightarrow$ 600 m³/s | <p> $\tau_{bs}^*$ $\approx$ 0.29, still dunes; </p> $\frac{\tau_{bs}}{\tau_b}$ $\approx$ 0.50 ; Z rises to ~2.0 --suspension weakening  |

## Scenario A - Finer Grain Size ($D_{50}$ 0.25 $\rightarrow$ 0.15 mm)
Change:
- $D_{50}$ to 0.15e-3 (finer sand)
- $D_{90}$ to 0.24e-3

Re-run all cells.

### Questions for Scenario A:
1. What bedform regime does vdBvG predict now? Did the point cross a boundary?
2. What happened to $\frac{\tau_{bs}}{\tau_b}$? Did the Einstein partition change, why or why not?
3. Z dropped dramatically. Is that mainly becasue $\omega_s$ changed, or because $u_*$ changed, or both? Quantify.
4. **Critical thinking**: if the bed is now in upper plane bed, what does the Einsten partition actually mean physically? Should you still apply it?

## Scenario B - Steeper Slope ($S$: 0.00008 $\rightarrow$ 0.0003)
Reset:
- $D_{50}$ to 0.25 mm
  
Change:
- $S$ to 0.0003

Re-run all cells.

### Questions for Scenario B:
1. What is $\tau_{bs}^*$ now and what bedform regime is predicted? Did this surprise you? Does slope affect what you expected?
2. $\tau_b$ increased substantially. What happened to $\frac{\tau_{bs}}{\tau_b}$? Did the partition fraction change, or just the absolute values?
3. Z decreased. Was that because $\omega_s$ changed or because $u_*$ changed? Explain the physical mechanism.
4. Compare Scenarios A and B: in both cases the bed left the dune regime, but for different reasons. Summarize the difference in one sentence.

## Scenario C - Lower Flow ($Q$ 1400 $\rightarrow$ 600 m³/s)
Reset:
- $S$ to 0.00008

Change:
- $Q$ to 600.0

Re-run all cells.

### Questions for Scenario C:
1. Does the river stay in the dune regime at lower flow?
2. Z increased to around 2.0. Is the river still actively suspending sediment at this discharge? What does Z $\approx$ 2 mean physically for the concentration profile shape?
3. The Rouse profile at Q=600 vs Q=1400: describe how the shape changes. Where in teh water column is the suspended sediment now concentrated?
4. Bringing it together: across all three scenarios, which single parameter change had the largest effect on Z and why?